本节我们将学习如何持久化保存模型状态、加载模型，并使用模型做预测。

In [1]:
import torch
import torchvision.models as models

### PART1.保存和加载模型权重

PyTorch 模型把训练得到的参数存放在一个内部字典对象，叫做state_dict（状态字典）。可以使用torch.save把它保存到磁盘。

In [4]:
# 加载预训练VGG16模型
model = models.vgg16(weights='IMAGENET1K_V1')
# 只保存模型权重（state_dict）到文件
torch.save(model.state_dict(), 'model_weights.pth')

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\81349/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100.0%


要加载模型权重，你需要先实例化出完全相同的模型对象，再调用load_state_dict()方法载入参数。

下方代码中设置weights_only=True，可以在反序列化加载文件时，仅执行加载权重必需的函数。加载权重时开启weights_only=True是官方推荐的最佳实践（防范恶意 pickle 文件安全风险）


In [5]:
# 不指定weights，创建一个随机初始化、未训练的VGG16空模型（结构必须和保存权重的模型完全一致）
model = models.vgg16()
# 读取pth权重文件，加载参数到模型
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
# 切换为评估推理模式（关闭Dropout、BN训练行为）
model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

### PART2.同时保存网络结构和权重

加载模型权重时，我们必须先实例化模型类，因为类定义了网络的结构。如果你希望把网络结构 + 权重一起保存，可以直接把完整 model 对象传给保存函数，而不是传入model.state_dict()：

In [6]:
torch.save(model, 'model.pth')

之后就可以按下面的方式加载完整模型。

正如模块保存与加载章节所述：只保存 state_dict 才是官方最佳实践。下面代码使用weights_only=False，因为直接保存整个 model 对象属于torch.save的旧时代遗留用法。

In [7]:
model = torch.load('model.pth', weights_only=False)